# Week 43, Day 6: Forge: Week 43 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w043/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
print("1.", np.log2(64), "bits per spike")
print("2.", 3 * np.sqrt(90 / 10), "cm")
print("3.", np.lcm(30, 42), "cm")

### Exercise: Part A: grids to place

Write place(w): the weighted sum of the 40 grid maps, minus its 97th percentile, rectified. Count the fields (connected regions above half the peak, using scipy.ndimage.label) and the distance of the peak from the target.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import label
r = np.random.default_rng(2006)
nb = 50; g = (np.arange(nb) + 0.5) / nb; X, Y = np.meshgrid(g, g, indexing="ij")

def grid(lam, ori, ph):
    k = 4 * np.pi / (np.sqrt(3) * lam); s = 0
    for a in [0, np.pi / 3, 2 * np.pi / 3]:
        s = s + np.cos(k * np.cos(a + ori) * (X - ph[0]) + k * np.sin(a + ori) * (Y - ph[1]))
    return np.maximum(0, (s + 1.5) / 4.5)

G = []
for m in range(4):
    lam = 0.3 * 1.4 ** m; ori = r.uniform(0, np.pi / 3)
    for i in range(10):
        G.append(grid(lam, ori, r.uniform(0, lam, 2)))
G = np.array(G)                                      # 40 x 50 x 50

def place(w):
    ...

def describe(p, target):
    lab, n = label(p > 0.5 * p.max())
    i, j = np.unravel_index(p.argmax(), p.shape)
    return n, np.hypot(g[i] - target[0], g[j] - target[1])

target = (0.62, 0.35); ti, tj = int(target[0] * nb), int(target[1] * nb)
p_sel, p_rand = place(G[:, ti, tj]), place(r.random(len(G)))
n_sel, dist_sel = describe(p_sel, target)
n_rand, _ = describe(p_rand, target)
print(f"selected weights: {n_sel} field(s), peak {100 * dist_sel:.1f} cm from the target;  random weights: {n_rand} fields")
fig, ax = plt.subplots(1, 2, figsize=(8, 4)); ax[0].imshow(p_sel.T, origin="lower"); ax[1].imshow(p_rand.T, origin="lower"); plt.show()

In [ ]:
import numpy as np
def test_selected():
    "selected weights give a single field within 10 cm of the target"
    assert n_sel == 1 and dist_sel < 0.1
def test_random():
    "random weights give more than one field"
    assert n_rand > 1

test_selected()
test_random()
print("All checks passed")

### Exercise: Part B: the rank-order test

Write rank_corr(a, b): the Pearson correlation of the ranks of a and b. Then rank_test(times): the rank correlation of field order with spike times and a two-sided p value from 1,000 shuffles of field order.

In [ ]:
N_c = 20; order = np.arange(N_c)

def make_event(kind, step=0.01, jitter=0.015):
    base = {"forward": order, "reverse": order[::-1], "random": r.permutation(N_c)}[kind] * step
    return base + jitter * r.normal(size=N_c)

def rank_corr(a, b):
    ...

def rank_test(times, n=1000):
    rho = rank_corr(order, times)
    null = np.array([rank_corr(r.permutation(N_c), times) for _ in range(n)])
    return rho, ...

rho_f, p_f = rank_test(make_event("forward"))
rho_r, p_r = rank_test(make_event("reverse"))
rho_c, p_c = rank_test(make_event("random"))
for nm, (rho, p) in {"forward": (rho_f, p_f), "reverse": (rho_r, p_r), "random": (rho_c, p_c)}.items():
    print(f"{nm:8s} rho = {rho:+.2f}, p = {p:.3f}")

In [ ]:
import numpy as np
def test_forward():
    "the forward event has a large positive rank correlation and p below 0.01"
    assert rho_f > 0.6 and p_f < 0.01
def test_reverse():
    "the reversed event is significant with negative correlation"
    assert rho_r < -0.6 and p_r < 0.01
def test_control():
    "the random event is not significant"
    assert p_c > 0.05

test_forward()
test_reverse()
test_control()
print("All checks passed")